# LangChain: Models, Prompts and Output Parsers

# 🤖 Sistema Multi-Agente para Geração e Avaliação de E-mails

### Engenharia de Prompt e Sistemas Multi-Agentes

---

## 👥 Grupo

| Integrante |
|---|
| **Rafael Lindoso** |
| **Gabriel Cavalcanti** |
| **Joellington Silva** |

## Get your [OpenAI API Key](https://platform.openai.com/account/api-keys)

In [ ]:
!pip install python-dotenv
!pip install openai

In [ ]:
from IPython.display import display, HTML
display(HTML(
"""
<a target="_blank" href="https://colab.research.google.com/github/pedrodiamel/agents-mini-course/blob/course/books/aula_03_model_prompt.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
"""
))

In [ ]:
import os
import openai
from utils import format_message, show_prompt


from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [5]:
# Set the model variable based on the current date
llm_model = "gpt-4o-mini"

## Chat API : OpenAI

Let's start with a direct API calls to OpenAI.

In [85]:
# Ref: https://platform.openai.com/docs/api-reference/chat/create

client = openai.OpenAI()

def get_completion(prompt, model=llm_model):
    messages = [
        {
        "role": "user",
        "content": prompt
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )

    return response.choices[0].message.content


In [13]:
client = openai.OpenAI()

with client.chat.completions.stream(
    model=llm_model,
    messages=[{"role": "user", "content": "Explique brevemente o que é aprendizado de máquina."}],
    temperature=0,
) as stream:
    for event in stream:
        if event.type == "message.delta":
            print(event.delta, end="", flush=True)

In [14]:
response = get_completion("Explique brevemente o que é aprendizado de máquina.")
print(response)

Aprendizado de máquina é um subcampo da inteligência artificial que se concentra no desenvolvimento de algoritmos e modelos que permitem que os computadores aprendam a partir de dados. Em vez de serem programados explicitamente para realizar uma tarefa, os sistemas de aprendizado de máquina identificam padrões e fazem previsões ou decisões com base em exemplos fornecidos. Isso é feito através de técnicas como regressão, classificação e agrupamento, e é amplamente utilizado em diversas aplicações, como reconhecimento de voz, recomendação de produtos, detecção de fraudes e muito mais. O objetivo é melhorar o desempenho do modelo à medida que mais dados se tornam disponíveis.


In [9]:
styles = ["formal and technical", "casual and friendly", "enthusiastic and persuasive", "concise and to the point", "storytelling and engaging"]
tones = ["confident", "empathetic", "urgent", "optimistic", "serious"]

In [10]:
prompt_task = """
Voce é um assistente de IA que ajuda a redigir emails corporativos. Para o contexto, hoje é {date}.

<Task>
Redigir um email com base no contexto fornecido.
</Task>

O email precisa ter a seguinte estrutura básica:
<Structure>
1. Saudação
2. Introdução ao problema
3. Apresentação da solução
4. Benefícios da solução
5. Chamada para ação (CTA)
6. Despedida
</Structure>
"""

prompt_context = """
<Context>
Escreva um email para um cliente corporativo com o seguinte contexto:
{customer_email}
O email deve ser escrito em um dos seguintes estilos: {style} e tons: {tone}.
</Context>
"""

prompt_instructions = """
<Instrutions>
Use o estilo e tom especificados para redigir o email.
Certifique-se de que o email seja claro, profissional e adequado ao público-alvo.
Não se esqueça de incluir um CTA (Call to Action) para agendar uma demonstração.
Não invente informações; baseie-se apenas no contexto fornecido.
Opcionalmente, use gatilhos mentais com autoridade, urgência, pertencimento, benefício se apropriado.
</Instrutions>
"""

prompt_references = """
<Example>
Por exemplo, um email formal e técnico com tom confiante pode ser:
Asunto: Apresentação de Solução Avançada de Visão Computacional para Inspeção Industrial
Prezado Sr. Silva,
Gostaria de apresentar nossa avançada solução de visão computacional projetada para otimizar processos de inspeção industrial.
Nossa tecnologia utiliza algoritmos de ponta para garantir precisão e eficiência, reduzindo custos operacionais.
Ficaria honrado em agendar uma demonstração para discutir como nossa solução pode beneficiar sua empresa.
Atenciosamente,
João Pereira
</Example>
"""

prompt = f"""{prompt_task}
{prompt_context}
{prompt_instructions}
{prompt_references}
"""


In [ ]:
show_prompt(prompt)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Voce é um assistente de IA que ajuda a redigir emails corporativos. Para o contexto, hoje é {date}.            │
│                                                                                                                 │
│  <Task>                                                                                                         │
│  Redigir um email com base no contexto fornecido.                                                               │
│  </Task>                                                                                                        │
│                                                                                                                 │
│  O email precisa ter a seguinte estrutura básica:                                                               │
│  <Structure>                                                                                                    │
│  1. Saudação                                                                                                    │
│  2. Introdução ao problema                                                                                      │
│  3. Apresentação da solução                                                                                     │
│  4. Benefícios da solução                                                                                       │
│  5. Chamada para ação (CTA)                                                                                     │
│  6. Despedida                                                                                                   │
│  </Structure>                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  <Context>                                                                                                      │
│  Escreva um email para um cliente corporativo com o seguinte contexto:                                          │
│  {customer_email}                                                                                               │
│  O email deve ser escrito em um dos seguintes estilos: {style} e tons: {tone}.                                  │
│  </Context>                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  <Instrutions>                                                                                                  │
│  Use o estilo e tom especificados para redigir o email.                                                         │
│  Certifique-se de que o email seja claro, profissional e adequado ao público-alvo.                              │
│  Não se esqueça de incluir um CTA (Call to Action) para agendar uma demonstração.                               │
│  Não invente informações; baseie-se apenas no contexto fornecido.                                               │
│  Opcionalmente, use gatilhos mentais com autoridade, urgência, pertencimento, benefício se apropriado.          │
│  </Instrutions>                                                                                                 │
│                                                                                                                 │
│                                                       

In [90]:
customer_email = """
Crie um email para um cliente corporativo apresentando nossa solução
de visão computacional para inspeção industrial. O estilo deve ser
casual e amigável, com tom empático e otimista. Inclua um CTA para
agendar uma demonstração.
"""

print(customer_email)


Crie um email para um cliente corporativo apresentando nossa solução
de visão computacional para inspeção industrial. O estilo deve ser
casual e amigável, com tom empático e otimista. Inclua um CTA para
agendar uma demonstração.



In [99]:
from datetime import datetime
def get_today_str() -> str:
    """Get current date in a human-readable format."""
    return datetime.now().strftime("%a %b %-d, %Y")

prompt.format(customer_email=customer_email, style=styles[0], tone=tones[0], date=get_today_str())
print(prompt)
response = get_completion(prompt)


Você é um especialista em análise de problemas e planejamento de comunicação.

Sua função é analisar cuidadosamente o problema apresentado pelo usuário
e elaborar um plano antes que qualquer texto seja escrito.

Você NÃO deve escrever o e-mail final.

Sua análise deve identificar:

1. Qual é o problema principal.
2. Qual é o contexto relevante.
3. Qual é o objetivo da comunicação.
4. Quem é o destinatário.
5. Qual tom de comunicação deve ser utilizado.
6. Quais informações precisam obrigatoriamente aparecer.
7. Quais informações devem ser evitadas.
8. Qual estratégia de comunicação é mais adequada.
9. Uma estrutura sugerida para o e-mail.

Se houver informações insuficientes, indique claramente quais informações
estão faltando.

Retorne um plano estruturado e objetivo que possa ser utilizado por outro
agente para escrever o e-mail.


Você é um especialista em comunicação profissional.

Sua função é escrever um e-mail com base:

- no problema apresentado pelo usuário;
- no plano produz

In [100]:
print(response)

Claro! Por favor, apresente o problema que você gostaria de analisar e planejar a comunicação.


## Chat API : LangChain

Let's try how we can do the same using LangChain.

In [23]:
!pip install --upgrade langchain
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.9
    Uninstalling langgraph-1.2.9:
      Successfully uninstalled langgraph-1.2.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 2.0 MB/s eta 0:00:00


### Model

In [24]:
from langchain_openai import ChatOpenAI

In [101]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(
    temperature=0.3,
    model=llm_model
    )
chat

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-openai': '1.5.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7bcf108a2000>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7bcf108a15e0>, root_client=<openai.OpenAI object at 0x7bcf108a1490>, root_asyn

In [102]:
from langchain_core.messages import HumanMessage

prompt.format(
    customer_email=customer_email,
    style=styles[0],
    tone=tones[0],
    date=get_today_str()
)

response = chat.invoke([HumanMessage(content=prompt)])
print("DONE")

DONE


In [103]:
from utils import format_message, show_prompt

show_prompt(prompt)
format_message([response])

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  Você é um especialista em análise de problemas e planejamento de comunicação.                                  │
│                                                                                                                 │
│  Sua função é analisar cuidadosamente o problema apresentado pelo usuário                                       │
│  e elaborar um plano antes que qualquer texto seja escrito.                                                     │
│                                                                                                                 │
│  Você NÃO deve escrever o e-mail final.                                                                         │
│                                                                                                                 │
│  Sua análise deve identificar:                                                                                  │
│                                                                                                                 │
│  1. Qual é o problema principal.                                                                                │
│  2. Qual é o contexto relevante.                                                                                │
│  3. Qual é o objetivo da comunicação.                                                                           │
│  4. Quem é o destinatário.                                                                                      │
│  5. Qual tom de comunicação deve ser utilizado.                                                                 │
│  6. Quais informações precisam obrigatoriamente aparecer.                                                       │
│  7. Quais informações devem ser evitadas.                                                                       │
│  8. Qual estratégia de comunicação é mais adequada.                                                             │
│  9. Uma estrutura sugerida para o e-mail.                                                                       │
│                                                                                                                 │
│  Se houver informações insuficientes, indique claramente quais informações                                      │
│  estão faltando.                                                                                                │
│                                                                                                                 │
│  Retorne um plano estruturado e objetivo que possa ser utilizado por outro                                      │
│  agente para escrever o e-mail.                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Você é um especialista em comunicação profissional.                                                            │
│                                                                                                                 │
│  Sua função é escrever um e-mail com base:                                                                      │
│                                                                                                                 │
│  - no problema apresentado pelo usuário;                                                                        │
│  - no plano produzido pelo Agente Reflexão.           

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│ Claro, por favor, forneça o problema que você gostaria que eu analisasse.                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Format output

In [120]:
from pydantic import BaseModel, Field

class EmailResponse(BaseModel):
    customer_email: str = Field(..., description="The email content generated for the customer.")
    tips: str = Field(..., description="Additional tips for improving the email.")
    chain_of_thought: str = Field(..., description="The reasoning process behind the email generation.")


In [121]:
email_writer_chain = (
    chat
    .with_structured_output(EmailResponse)
)

prompt.format(
    customer_email=customer_email,
    style=styles[0],
    tone=tones[0],
    date=get_today_str()
)

response = email_writer_chain.invoke([HumanMessage(content=prompt)])
print("DONE")

DONE


In [122]:
response

EmailResponse(customer_email='Assunto: Solicitação de Atualização sobre o Projeto\n\nPrezado [Nome do Destinatário],\n\nEspero que esteja bem.\n\nEstou entrando em contato para solicitar uma atualização sobre o andamento do projeto [Nome do Projeto]. Como sabemos, a data de entrega se aproxima e gostaria de entender como estão as etapas atuais e se há alguma pendência que possamos resolver juntos.\n\nAgradeço pela atenção e fico no aguardo de sua resposta.\n\nAtenciosamente,\n\n[Seu Nome]  \n[Seu Cargo]  \n[Seu Contato]  \n[Nome da Empresa]', tips='Certifique-se de personalizar o e-mail com o nome do destinatário e detalhes específicos do projeto para torná-lo mais relevante e engajador.', chain_of_thought='1. O problema principal é a necessidade de uma atualização sobre o andamento de um projeto específico.  \n2. O contexto relevante é que a data de entrega do projeto está se aproximando, o que aumenta a urgência da solicitação.  \n3. O objetivo da comunicação é obter informações sobr

In [123]:
print("Customer Email:")
print(response.customer_email)
print("\nTips:")
print(response.tips)
print("\nChain of Thought:")
print(response.chain_of_thought)

Customer Email:
Assunto: Solicitação de Atualização sobre o Projeto

Prezado [Nome do Destinatário],

Espero que esteja bem.

Estou entrando em contato para solicitar uma atualização sobre o andamento do projeto [Nome do Projeto]. Como sabemos, a data de entrega se aproxima e gostaria de entender como estão as etapas atuais e se há alguma pendência que possamos resolver juntos.

Agradeço pela atenção e fico no aguardo de sua resposta.

Atenciosamente,

[Seu Nome]  
[Seu Cargo]  
[Seu Contato]  
[Nome da Empresa]

Tips:
Certifique-se de personalizar o e-mail com o nome do destinatário e detalhes específicos do projeto para torná-lo mais relevante e engajador.

Chain of Thought:
1. O problema principal é a necessidade de uma atualização sobre o andamento de um projeto específico.  
2. O contexto relevante é que a data de entrega do projeto está se aproximando, o que aumenta a urgência da solicitação.  
3. O objetivo da comunicação é obter informações sobre o status do projeto e identific

### Exercisio

Para aprimorar os resultados obtidos vamos a criar um sistemas multi-agente para conseguir fazer uma análise do problema (reflexão), escrever o email a partir deste plano (escrita) e finalmente avaliar os resultados obtidos (avaliação).

- Step 1: Crie os três agentes (Agente reflexão, Agente escrita, Agente avaliação)
- Step 2: Crie um pipeline (chain) para gerar os resultados
- Step 3: Avalie e intere sobre os resultados obtidos


Exercício

In [113]:
prompt_reflexao = """
Você é um especialista em análise de problemas e planejamento de comunicação.

Sua função é analisar cuidadosamente o problema apresentado pelo usuário
e elaborar um plano antes que qualquer texto seja escrito.

Você NÃO deve escrever o e-mail final.

Sua análise deve identificar:

1. Qual é o problema principal.
2. Qual é o contexto relevante.
3. Qual é o objetivo da comunicação.
4. Quem é o destinatário.
5. Qual tom de comunicação deve ser utilizado.
6. Quais informações precisam obrigatoriamente aparecer.
7. Quais informações devem ser evitadas.
8. Qual estratégia de comunicação é mais adequada.
9. Uma estrutura sugerida para o e-mail.

Se houver informações insuficientes, indique claramente quais informações
estão faltando.

Retorne um plano estruturado e objetivo que possa ser utilizado por outro
agente para escrever o e-mail.
"""

prompt_escrita = """
Você é um especialista em comunicação profissional.

Sua função é escrever um e-mail com base:

- no problema apresentado pelo usuário;
- no plano produzido pelo Agente Reflexão.

Siga rigorosamente o plano recebido.

O e-mail deve:

1. Ser claro.
2. Ser objetivo.
3. Ter linguagem profissional.
4. Possuir assunto.
5. Possuir saudação adequada.
6. Apresentar o problema de maneira clara.
7. Apresentar as informações relevantes.
8. Ter uma solicitação ou objetivo claro.
9. Ter uma conclusão adequada.

Não invente informações que não estejam disponíveis.

Se alguma informação estiver faltando, escreva de forma que o e-mail
continue coerente sem inventar dados.

Retorne somente o e-mail final.
"""

prompt_avaliacao = """
Você é um avaliador crítico especializado em comunicação profissional.

Sua função é avaliar um e-mail produzido por outro agente.

Você deve comparar:

- o problema original;
- o plano produzido pelo Agente Reflexão;
- o e-mail produzido pelo Agente Escrita.

Avalie:

1. Clareza.
2. Objetividade.
3. Adequação ao destinatário.
4. Tom profissional.
5. Atendimento ao objetivo.
6. Cobertura das informações importantes.
7. Coerência com o plano.
8. Ausência de informações inventadas.
9. Qualidade geral do texto.

Atribua uma nota de 0 a 10.

Se a nota for menor que 8, indique claramente:
- quais problemas foram encontrados;
- como o e-mail pode ser melhorado;
- quais alterações devem ser feitas.

Se a nota for 8 ou superior, considere o resultado aprovado.

Retorne obrigatoriamente neste formato:

NOTA: X/10

STATUS: APROVADO ou REVISAR

PROBLEMAS:
- problema 1
- problema 2

SUGESTÕES:
- sugestão 1
- sugestão 2
"""

prompt = f"""{prompt_reflexao}
{prompt_escrita}
{prompt_avaliacao}
"""
show_prompt(prompt)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  Você é um especialista em análise de problemas e planejamento de comunicação.                                  │
│                                                                                                                 │
│  Sua função é analisar cuidadosamente o problema apresentado pelo usuário                                       │
│  e elaborar um plano antes que qualquer texto seja escrito.                                                     │
│                                                                                                                 │
│  Você NÃO deve escrever o e-mail final.                                                                         │
│                                                                                                                 │
│  Sua análise deve identificar:                                                                                  │
│                                                                                                                 │
│  1. Qual é o problema principal.                                                                                │
│  2. Qual é o contexto relevante.                                                                                │
│  3. Qual é o objetivo da comunicação.                                                                           │
│  4. Quem é o destinatário.                                                                                      │
│  5. Qual tom de comunicação deve ser utilizado.                                                                 │
│  6. Quais informações precisam obrigatoriamente aparecer.                                                       │
│  7. Quais informações devem ser evitadas.                                                                       │
│  8. Qual estratégia de comunicação é mais adequada.                                                             │
│  9. Uma estrutura sugerida para o e-mail.                                                                       │
│                                                                                                                 │
│  Se houver informações insuficientes, indique claramente quais informações                                      │
│  estão faltando.                                                                                                │
│                                                                                                                 │
│  Retorne um plano estruturado e objetivo que possa ser utilizado por outro                                      │
│  agente para escrever o e-mail.                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Você é um especialista em comunicação profissional.                                                            │
│                                                                                                                 │
│  Sua função é escrever um e-mail com base:                                                                      │
│                                                                                                                 │
│  - no problema apresentado pelo usuário;                                                                        │
│  - no plano produzido pelo Agente Reflexão.           

In [157]:
from pydantic import BaseModel, Field

class ReflexaoResponse(BaseModel):
    problema: str = Field(..., description="Qual é o problema principal.")
    contexto: str = Field(..., description="Qual é o contexto relevante.")
    objetivo: str = Field(..., description="Qual é o objetivo da comunicação.")
    destinatario: str = Field(..., description="Quem é o destinatário.")
    tom: str = Field(..., description="Qual tom de comunicação deve ser utilizado.")
    informacao: str = Field(..., description="Quais informações precisam obrigatoriamente aparecer.")
    evitar: str = Field(..., description="Quais informações devem ser evitadas.")
    estrategia: str = Field(..., description="Qual estratégia de comunicação é mais adequada.")
    estrutura: str = Field(..., description="Uma estrutura sugerida para o e-mail.")


In [158]:
reflexao_writer_chain = (
    chat
    .with_structured_output(ReflexaoResponse)
)

reflexao = f"{prompt_reflexao} /n {response}"

reflexao_response = reflexao_writer_chain.invoke([HumanMessage(content=reflexao)])

print(reflexao_response)

problema='Necessidade de uma atualização sobre o andamento de um projeto específico.' contexto='A data de entrega do projeto está se aproximando, aumentando a urgência da solicitação.' objetivo='Obter informações sobre o status do projeto e identificar possíveis pendências.' destinatario='Uma pessoa envolvida no projeto, possivelmente um colega ou superior.' tom='Profissional e respeitoso, mantendo a cordialidade.' informacao='Nome do projeto, solicitação de atualização e a urgência devido à data de entrega.' evitar='Críticas ou reclamações sobre o andamento do projeto.' estrategia='Abordagem direta e respeitosa, focando na colaboração.' estrutura='Saudação, introdução do motivo do contato, solicitação clara, agradecimento e fechamento cordial.'


In [118]:
excrita_writer_chain = (
    chat
    .with_structured_output(EmailResponse)
)

escrita = f"{prompt_escrita} /n {reflexao_response} /n {response}"

escrita_response = excrita_writer_chain.invoke([HumanMessage(content=escrita)])

print(escrita_response)

customer_email='Assunto: Solicitação de Atualização sobre o Projeto\n\nPrezado [Nome do Destinatário],\n\nEspero que você esteja bem.\n\nEstou entrando em contato para solicitar uma atualização sobre o andamento do projeto [Nome do Projeto]. É importante para nossa equipe entender o status atual, especialmente em relação aos prazos e entregas.\n\nAgradeço se puder compartilhar as informações mais recentes e se há algo que possamos fazer para ajudar a avançar com o projeto.\n\nAguardo sua resposta.\n\nAtenciosamente,\n\n[Seu Nome]  \n[Seu Cargo]  \n[Seu Contato]  \n[Nome da Empresa]' tips='Considere adicionar um prazo para a resposta, se aplicável, para enfatizar a urgência da solicitação.' chain_of_thought='O e-mail foi estruturado para ser claro e objetivo, abordando diretamente a solicitação de atualização sobre o projeto. O tom é profissional e respeitoso, adequado para a comunicação entre colegas de trabalho. As informações relevantes foram incluídas, como o nome do projeto e a sol

In [148]:
from pydantic import BaseModel, Field

class AvaliacaoResponse(BaseModel):
    avaliacao: str = Field(..., description="Resumo da avaliação gerada.")
    nota: int = Field(..., description="A score from 1 to 10 evaluating the quality of the email.")
    status: str = Field(..., description="Status of the email generation process. ['APROVADO' or 'REVISAR']")

    def escreva(self):
      print(f"Nota: {self.nota}")
      print(f"Status: {self.status}")
      print(f"Avaliação: {self.avaliacao}")

In [127]:
avaliacao_writer_chain = (
    chat
    .with_structured_output(AvaliacaoResponse)
)

avaliacao = f"Problema original: {response}, plano produzido: {reflexao_response} /n E-mail produzido: {escrita_response}"
avaliacao_response = avaliacao_writer_chain.invoke([HumanMessage(content=avaliacao)])
print(avaliacao_response)

avaliacao='O e-mail foi bem estruturado e mantém um tom profissional e respeitoso. A solicitação de atualização sobre o projeto é clara e direta, com ênfase na importância das informações para o planejamento da equipe. A inclusão de um pedido para saber como a equipe pode ajudar também demonstra colaboração. No entanto, a sugestão de adicionar um prazo para a resposta poderia aumentar a urgência da solicitação, mas isso não compromete a qualidade geral do e-mail.' nota=9 status='APROVADO'


In [159]:
email = """
Assunto: Pedido de Atualização sobre o Projeto

Olá [Nome do Destinatário],

Espero que este vos encontre breve.

Estou entrando em contato para pedir encarecidamente uma atualização sobre a evolução do projeto [Nome do Projeto]. Sinto que ele está demasiadamente atrasado e isto afeta meu humor. É primordial para nossa equipe entender o estágio atual da obra, especialmente em relação aos atrazos ou adianto de entregas.

Peço para compartilhar as informações atuais o mais rápido possível, pois da ultima vez demorou bastante e o chefe não gostou. Claro, se tem algo que possa fazer para ajudar a andar com o projeto.

Por obsequio, atenda esta solicitação o mais breve, mas sem querer apressar as coisas

Espero sua resposta.

Cordialmente,

[Seu Nome]
[Seu Cargo]
[Seu Contato]
[Nome da Empresa]
"""


In [160]:
def executar_pipeline_iterativo(problema, max_iteracoes=3):
  reflexao_response = None
  escrita_response = None
  escrita_response = problema
  avaliacao_writer_chain = (
    chat
    .with_structured_output(AvaliacaoResponse)
  )

  avaliacao = f"Problema original: {problema} /n E-mail produzido: {escrita_response}"
  avaliacao_response = avaliacao_writer_chain.invoke([HumanMessage(content=avaliacao)])
  avaliacao_response.escreva()


  for tentativa in range(max_iteracoes):
    print(f"\n{'='*60}")
    print(f"TENTATIVA {tentativa + 1}")
    print(f"{'='*60}")
    reflexao_writer_chain = (
      chat
      .with_structured_output(ReflexaoResponse)
    )

    reflexao = f"{prompt_reflexao} /n {escrita_response}"

    reflexao_response = reflexao_writer_chain.invoke([HumanMessage(content=reflexao)])

    excrita_writer_chain = (
      chat
      .with_structured_output(EmailResponse)
    )

    escrita = f"{prompt_escrita} /n {reflexao_response} /n {escrita_response}"

    escrita_response = excrita_writer_chain.invoke([HumanMessage(content=escrita)])

    avaliacao_writer_chain = (
      chat
      .with_structured_output(AvaliacaoResponse)
    )

    avaliacao = f"Problema original: {problema}, plano produzido: {reflexao_response} /n E-mail produzido: {escrita_response}"
    avaliacao_response = avaliacao_writer_chain.invoke([HumanMessage(content=avaliacao)])
    avaliacao_response.escreva()

    if avaliacao_response.status == 'APROVADO':
      print(escrita_response.customer_email)
      break



In [161]:
executar_pipeline_iterativo(email)

Nota: 6
Status: REVISAR
Avaliação: O e-mail apresenta um tom respeitoso e educado, mas a repetição de algumas expressões e a forma como a urgência é abordada podem ser melhoradas. A mensagem é clara em seu pedido, mas o uso de frases como 'isto afeta meu humor' pode ser interpretado como um desvio do foco profissional. Além disso, a frase 'sem querer apressar as coisas' contradiz o pedido de rapidez, o que pode causar confusão. Uma reformulação para um tom mais objetivo e profissional seria benéfica.

TENTATIVA 1
Nota: 9
Status: APROVADO
Avaliação: O e-mail produzido é claro, profissional e mantém um tom amigável e colaborativo. A estrutura segue o plano proposto, abordando todos os pontos necessários, como a solicitação de atualização, a importância da informação e a oferta de ajuda. Além disso, evita qualquer linguagem negativa que possa desmotivar a equipe. A personalização dos campos entre colchetes é uma boa prática recomendada. No geral, o e-mail está bem elaborado e atende ao ob

In [156]:
print(escrita_response.customer_email)

Assunto: Solicitação de Atualização sobre o Projeto

Prezado [Nome do Destinatário],

Espero que você esteja bem.

Estou entrando em contato para solicitar uma atualização sobre o andamento do projeto [Nome do Projeto]. É importante para nossa equipe entender o status atual, especialmente em relação aos prazos e entregas.

Agradeço se puder compartilhar as informações mais recentes e se há algo que possamos fazer para ajudar a avançar com o projeto.

Aguardo sua resposta.

Atenciosamente,

[Seu Nome]  
[Seu Cargo]  
[Seu Contato]  
[Nome da Empresa]
